 # Intrucciones

 **Entrenar un modelo con YOLOv8 para OBJECT DETECTOR personalizado
 Debe Cargar el dataset en el Directorio de Google Colab
 **

Configurar el tipo de entro GPU o TPU


#Instalamos las librerias de utralytics y open-python

In [ ]:
!pip install ultralytics
# Si requiere debe instalr opencv-python
# Requiere instalar pytorch, generalmente lo tienes ya instalado en google Colab.

In [ ]:
%pip show ultralytics

In [ ]:
!nvidia-smi

#Importar todos las bibliotecas requirdas en este proyecto

In [ ]:
import torch
import os
import cv2
import shutil
import zipfile
import requests
import random
from ultralytics import YOLO
from IPython.display import Image, clear_output

#Pasos para aportar un nuevo conjunto de datos

1. **Recopilar imágenes:** Reúna las imágenes que pertenecen al conjunto de datos. Pueden proceder de diversas fuentes, como bases de datos públicas o tu propia colección.

2. **Anotar imágenes:** Anote estas imágenes con cuadros delimitadores, segmentos o puntos clave, en función de la tarea.

3. **Exportar anotaciones:** Convierte estas anotaciones en YOLO *.txt que admite Ultralytics .

4. **Organizar el conjunto de datos:** Organice su conjunto de datos en la estructura de carpetas correcta. Debe tener train/ y val/ directorios de nivel superior, y dentro de cada uno, un images/ y labels/ subdirectorio.


```text
dataset/
├── train/
│   ├── images/
│   └── labels/
├── valid/
│    ├── images/
│    └── labels/
└── test/
    ├── images/
    └── labels/
    
```

5. **Crear un dataset.yaml Archivo:** En el directorio raíz de su conjunto de datos, cree un archivo data.yaml que describe el conjunto de datos, las clases y otra información necesaria.

6. **Optimizar imágenes (opcional): **Si desea reducir el tamaño del conjunto de datos para un procesamiento más eficiente, puede optimizar las imágenes utilizando el código siguiente. No es obligatorio y no se recomienda.

7. **Comprimir conjunto de datos:** Comprime toda la carpeta del conjunto de datos en un archivo zip.

#1. Descargar los archivos con sus anotaciones en Yolo

Cargue el zip con las imagenes y los labels co la estructura indicada


El siguiente codigo borrar la carpeta si usted ejecutó el codigo anteriormente.


In [ ]:
!rm -rf /content/dataset/

In [ ]:
# Definir rutas base
base_path = "/content"
zip_path = os.path.join(base_path, "PPE Detection.v1-base_ver.yolov8.zip") # UNombre el zip decargado con imagenes y labels


def extract_zip(zip_file, dest_folder):
    dataset_folder = os.path.join(dest_folder, "dataset")  # Extraer en "dataset/"
    os.makedirs(dataset_folder, exist_ok=True)  # Asegurar que la carpeta existe
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(dataset_folder)  # Extraer en dataset/
    print(f"Contenido extraído en {dataset_folder}: {os.listdir(dataset_folder)}")

# Ejecutar procesos
extract_zip(zip_path, base_path)


# Si requiere espacion en la carpeta puede borrar el archivo zip. De lo contrario puede mantenerlo.

In [ ]:
#borrar el zip
!rm -rf "/content/Hard Hat Detector.v1i.yolov8.zip"

Vamos a comprobar que las etiquetas de la parte train, solo para verficiar que quedaron bien las descargas y la descompresión.

In [ ]:
labels_val_path = "/content/dataset/train/labels"  # Ajusta la ruta según tu dataset

if os.path.exists(labels_val_path):
    label_files = [f for f in os.listdir(labels_val_path) if f.endswith(".txt")]
    if len(label_files) == 0:
        print("⚠️ No se encontraron etiquetas en:", labels_val_path)
    else:
        print(f"✅ Se encontraron {len(label_files)} archivos de etiquetas.")
else:
    print("⚠️ La carpeta de etiquetas no existe.")

Vamos a validar que tenemos todos los archivos y carpetas organizados de la manera como ultralytics y el data.yaml requiere para hacer el fin tuning del modelo yolo.

In [ ]:
import os

base_path = "/content/dataset"
subdirs = ["train/images", "train/labels", "valid/images", "valid/labels", "test/images", "test/labels"]

for subdir in subdirs:
    full_path = os.path.join(base_path, subdir)
    if not os.path.exists(full_path):
        print(f"🚨 ERROR: No se encontró la carpeta {full_path}")
    else:
        print(f"✅ {full_path} encontrado, contiene {len(os.listdir(full_path))} archivos")

### Edite el Archivo data.yaml para el proyecto, este es un ejemplo.  Combielo a sus necesidades.
```text
path: /content/dataset
train: train/images
val: valid/images
test: test/images

nc: 5
names: ['botas', 'guantes', 'casco', 'trabajador', 'chaleco']

```

In [ ]:
model = YOLO("yolov8n.pt")

# **Carga  Tensorboard para visualizar las métricas de entrenamiento**

In [ ]:
#@title Select YOLOv8 🚀 logger {run: 'auto'}
logger = 'TensorBoard' #@param ['ClearML', 'Comet', 'TensorBoard']

if logger == 'ClearML':
  %pip install -q clearml
  import clearml; clearml.browser_login()
elif logger == 'Comet':
  %pip install -q comet_ml
  import comet_ml; comet_ml.init()
elif logger == 'TensorBoard':
  %reload_ext tensorboard
  %tensorboard --logdir runs/detect/train8

Podemos pasar los siguientes argumentos en el comando de entrenamiento:

imgsz: Define el tamaño de la imagen de entrada.

batch: Determina el tamaño del lote.

epochs: Define el número de épocas de entrenamiento.

data: Establece la ruta a nuestro archivo YAML.

name: Nombre del experimento.

project: Nombre del proyecto.

model: Especifica la ruta a los pesos del modelo YOLOv8 preentrenado. Puedes usar los pesos de cualquier modelo que quieras entrenar:

yolov8n.pt

yolov8s.pt

yolov8m.pt

yolov8l.pt

yolov8x.pt

![imagen](https://learnopencv.com/wp-content/uploads/2023/01/yolov8-object-detection-models.png)

También puedes usar modelos en formato .yaml para entrenar desde cero, por ejemplo: yolov8n.yaml.

NOTA: Hay muchos más parámetros que puedes configurar. Lee más sobre ellos aquí: Documentación de Ultralytics o en el archivo default.yaml: default.yaml en GitHub.

In [ ]:
model.train(data="/content/dataset/data.yaml", epochs=50, imgsz=640,batch=16)
# batch=16, epochs=100, imgsz=640, workers=1

Entrenar desde un scratch. Si el entramienamiento se detiene me manera anormal, los resultados de las epocas se van guardando en la carpeta runs.
Si desea reiniciar en donde el entramiento tuvo el scratch, debe usar este codigo.

```python
from ultralytics import YOLO

# Load a model
#model = YOLO("yolov8s.yaml")  # build a new model from scratch
model = YOLO("/runs/detect/train/weights/last.pt")  # load a pretrained model (recommended for training)

# Train the model
model.train(data="data.yaml", batch=16, epochs=100, imgsz=640, workers=1)
```


#Revisar las métcias del modelo

In [ ]:
metrics = model.val()
print(metrics)

# **Testear**

Vamos a testear el modelo, con una imagen de test (no con los datos valid, porque estos fueron usandos en el entrenamiento). Seleccionamis una imagen para testear.



In [ ]:

# Cargar la imagen
image_path = "/content/dataset/test/images/Video1_173_jpg.rf.8d2b0222932dd1b5ccbedc79c7800fc9.jpg"
# Esta es otra imange /content/dataset/test/images/006070_jpg.rf.cb5fefd1ba716d1560b6326003732e4c.jpg"
image = cv2.imread(image_path)
results = model.predict(image_path, save=True, imgsz=640)

# Guardar la imagen
cv2.imwrite("imagen.jpg", image)

In [ ]:
results[0].show()

# Vamos a predecir todos las imagenes de test


In [ ]:
#Hagamos la prediccion de todas las images que están en test

# Definir la ruta a la carpeta de imágenes de prueba
test_images_dir = "/content/dataset/test/images"

# Iterar sobre todas las imágenes en la carpeta de prueba
for filename in os.listdir(test_images_dir):
    if filename.endswith(('.jpg', '.jpeg', '.png')):  # Ajusta las extensiones si es necesario
        image_path = os.path.join(test_images_dir, filename)
        results = model.predict(image_path, save=True, imgsz=640)  # Realizar la predicción
        print(f"Predicción realizada para: {filename}")
        # Puedes acceder a las predicciones a través de results[0].boxes, results[0].probs, etc.
        # Si deseas visualizar las imágenes con las predicciones:
        results[0].show()

**Deteccion de video**

```python
# Ruta del video de prueba
video_source = "/mydrive/mask_test_videos/testaxle2.mp4"

# Realizar la predicción
results = model.predict(source=video_source, conf=0.3, save=True)

print("Predicción completada y resultados guardados.")       
```

#CONVERTIR A TFLITE, ONNX, TORCHSCRIPT Y MODELO TENSORFLOW

Exporta un modelo YOLOv8 a cualquier formato compatible utilizando el argumento format, por ejemplo: format=onnx.



<table>
<thead>
<tr>
  <th>Format</th>
  <th><code>format=...</code></th>
  <th>Model</th>
</tr>
</thead>
<tbody>
<tr>
  <td>PyTorch</td>
  <td>-</td>
  <td>yolov8n.pt</td>
</tr>
<tr>
  <td>TorchScript</td>
  <td><code>torchscript</code></td>
  <td>yolov8n.torchscript</td>
</tr>
<tr>
  <td>ONNX</td>
  <td><code>onnx</code></td>
  <td>yolov8n.onnx</td>
</tr>
<tr>
  <td>OpenVINO</td>
  <td><code>openvino</code></td>
  <td>yolov8n_openvino_model/</td>
</tr>
<tr>
  <td>TensorRT</td>
  <td><code>engine</code></td>
  <td>yolov8n.engine</td>
</tr>
<tr>
  <td>CoreML</td>
  <td><code>coreml</code></td>
  <td>yolov8n.mlmodel</td>
</tr>
<tr>
  <td>TensorFlow SavedModel</td>
  <td><code>saved_model</code></td>
  <td>yolov8n_saved_model/</td>
</tr>
<tr>
  <td>TensorFlow GraphDef</td>
  <td><code>pb</code></td>
  <td>yolov8n.pb</td>
</tr>
<tr>
  <td>TensorFlow Lite</td>
  <td><code>tflite</code></td>
  <td>yolov8n.tflite</td>
</tr>
<tr>
  <td>TensorFlow Edge TPU</td>
  <td><code>edgetpu</code></td>
  <td>yolov8n_edgetpu.tflite</td>
</tr>
<tr>
  <td>TensorFlow.js</td>
  <td><code>tfjs</code></td>
  <td>yolov8n_web_model/</td>
</tr>
<tr>
  <td>PaddlePaddle</td>
  <td><code>paddle</code></td>
  <td>yolov8n_paddle_model/</td>
</tr>
</tbody>
</table>

In [ ]:
model.export(format="onnx")  # Exportar a ONNX
model.export(format="torchscript")  # Exportar a TorchScript


In [ ]:
from ultralytics import YOLO

model.export(format="onnx",opset=12)  # export the model to ONNX format

In [ ]:
# Exportar a formato TensorFlow SavedModel
model.export(format="saved_model")

print("Modelo exportado a TensorFlow SavedModel.")

In [ ]:
import os

export_path = "runs/detect/train/weights/"
files = os.listdir(export_path)

print("Archivos exportados en:", export_path)
print(files)

In [ ]:
# Exportar a formato TensorFlow Lite (TFLite)
model.export(format="tflite")

print("Modelo exportado a TensorFlow Lite (TFLite).")


Mover de ubicación los modelos salvados

In [ ]:
#descargar toda la carpeta dataset  del gogle colab al escritorio del pc

from google.colab import files
import os
import shutil

# Define the source directory (where your dataset folder is)
source_dir = "/content/dataset"

# Define the destination directory on your local machine (your Desktop)
destination_dir = "C:/Users/USUARIO/Documents"


# Create a zip file of the dataset folder
shutil.make_archive(source_dir, 'zip', source_dir)


# Download the zip file to your local machine
files.download(f'{source_dir}.zip')
